# Student Dropout Prediction - Preprocessing

This notebook prepares the fixed MVP dataset used by the modeling notebook and app. It removes `Enrolled`, keeps only the 10 MVP features, and saves both numeric and model-ready processed datasets.

In [ ]:
# Import libraries used for preprocessing.
from pathlib import Path
import json

import pandas as pd

## Load Data

Resolve project paths, load the raw dataset, and load the MVP feature scope from `feature_config.json`.

In [ ]:
# Resolve project paths.
current_path = Path.cwd()

if (current_path / "data" / "raw" / "dataset.csv").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
FEATURE_CONFIG_PATH = PROJECT_ROOT / "app" / "feature_config.json"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed.csv"
MVP_NUMERIC_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mvp_features_numeric.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", DATA_PATH)
print("Feature config path:", FEATURE_CONFIG_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)
print("Numeric MVP data path:", MVP_NUMERIC_DATA_PATH)

In [ ]:
# Load raw data and feature config.
df = pd.read_csv(DATA_PATH)

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

candidate_mvp_features = feature_config["features"]

print("Raw dataset shape:", df.shape)
print("Target distribution:")
print(df["Target"].value_counts())
print("\nMVP features:", candidate_mvp_features)

## Filter to Final MVP Scope

The numeric MVP dataset keeps original encoded feature values. `Enrolled` is removed, while target labels remain readable (`Graduate` / `Dropout`) for inspection.

In [ ]:
# Keep fixed MVP features and remove Enrolled.
selected_columns = candidate_mvp_features + ["Target"]
missing_columns = [col for col in selected_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_mvp_numeric = df[selected_columns].copy()
df_mvp_numeric = df_mvp_numeric[df_mvp_numeric["Target"] != "Enrolled"].copy()

print("Numeric MVP dataset shape:", df_mvp_numeric.shape)
print("Target distribution after removing Enrolled:")
print(df_mvp_numeric["Target"].value_counts())

df_mvp_numeric.head()

## Save Numeric and Processed Data

`mvp_features_numeric.csv` keeps readable target labels. `processed.csv` uses the same numeric features but encodes the target as `Graduate = 0`, `Dropout = 1` for modeling.

In [ ]:
# Save numeric MVP dataset and model-ready processed dataset.
target_mapping = {
    "Graduate": 0,
    "Dropout": 1
}

df_processed = df_mvp_numeric.copy()
df_processed["Target"] = df_processed["Target"].map(target_mapping)

if df_processed["Target"].isna().any():
    raise ValueError("Target mapping created missing values.")

MVP_NUMERIC_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_mvp_numeric.to_csv(MVP_NUMERIC_DATA_PATH, index=False)
df_processed.to_csv(PROCESSED_DATA_PATH, index=False)

print("Saved numeric MVP dataset to:", MVP_NUMERIC_DATA_PATH)
print("Saved processed dataset to:", PROCESSED_DATA_PATH)
print("Processed dataset shape:", df_processed.shape)
print("Encoded target distribution:")
print(df_processed["Target"].value_counts().sort_index())

## Modeling Preprocessing Design

The modeling notebook applies these transformations:

- Binary flags (`Displaced`, `Educational special needs`, `Gender`, `International`): passthrough because they are already 0/1.
- `Age at enrollment`: `RobustScaler` because EDA shows right skew and high-age outliers.
- `Marital status`, `Course`, `Previous qualification`: one-hot encoding because they are nominal with manageable cardinality.
- Parents' qualification features: target encoding because they have many sparse categories.
- Models use class balancing and threshold tuning because Dropout recall is the priority.